# FunctionGemma 270M × TinyCeNN Memory Fusion — Sequential Acceptance

This is the FunctionGemma counterpart of the SmolLM2 Memory Fusion sequential experiment. It uses `vtava/functiongemma-270m-it-simple-tool-calling` as both teacher and starting student.

Gemma3 has hybrid attention. V1 **only replaces the original full-attention anchor layers** (normally 5, 11, 17) and intentionally keeps all sliding-window layers unchanged. Each replacement must pass NMSE, cosine, incremental NLL, and cumulative NLL gates before the next anchor is touched. Failed layers are saved and resumed instead of discarded.

After each run, the notebook reconstructs the accepted snapshot and compares its tool-calling outputs with the original FunctionGemma model.


In [ ]:
import os, sys, subprocess, shutil, shlex
from pathlib import Path

assert subprocess.run(["nvidia-smi"], check=False).returncode == 0, "Enable a GPU runtime in Colab."
REPO = Path("/content/TinyCeNN-LM")
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "datasets>=3,<5", "huggingface_hub>=0.36", "pytest", "pandas"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"], check=True)
for p in (str(REPO), str(REPO / "src")):
    if p not in sys.path: sys.path.insert(0, p)
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO), str(REPO / "src")])
import tinycenn_lm
print("✅ tinycenn_lm:", tinycenn_lm.__file__)
print("Git commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
import json, torch
from huggingface_hub import HfApi
from transformers import AutoConfig
from google.colab import drive

MODEL_ID = "vtava/functiongemma-270m-it-simple-tool-calling"
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
FEATURE_DIM = 32
MEMORY_RANK = 64
CONTEXT = 128
MAX_ROUNDS_PER_RUN = 4
RESET_PROGRESS = False  # set True only when you deliberately want a fresh experiment

drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/TinyCeNN-LM/functiongemma-memory-fusion-sequential-r64")
if RESET_PROGRESS and OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "last_colab_run.log"

cfg = AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION).get_text_config(decoder=True)
full = [i for i, kind in enumerate(cfg.layer_types) if kind == "full_attention"]
sliding = [i for i, kind in enumerate(cfg.layer_types) if kind == "sliding_attention"]
print(json.dumps({"model": MODEL_ID, "revision": MODEL_REVISION, "layers": cfg.num_hidden_layers, "full_attention": full, "sliding_attention": sliding, "output": str(OUT)}, indent=2))
assert cfg.model_type == "gemma3_text"
assert full, "No full-attention anchors found"


## Preflight
Run the Gemma3 Memory Fusion wrapper tests before spending GPU time.


In [ ]:
env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_gemma3_memory_fusion.py"], cwd=REPO, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError(f"preflight failed: {r.returncode}")
print("✅ preflight passed")


## Train / resume sequential full-attention replacement

The same folder is reused on every run. If a layer misses the acceptance gate, rerun this cell or reopen the notebook later; its current weights resume exactly where they stopped.


In [ ]:
cmd = [
    sys.executable, "-u", str(REPO / "scripts" / "train_functiongemma_memory_fusion_sequential.py"),
    "--base-model", MODEL_ID,
    "--model-revision", MODEL_REVISION,
    "--output-dir", str(OUT),
    "--feature-dim", str(FEATURE_DIM),
    "--memory-rank", str(MEMORY_RANK),
    "--context-length", str(CONTEXT),
    "--probe-context", str(CONTEXT),
    "--seed", "73",
    "--min-layer-steps", "50",
    "--max-layer-steps", "300",
    "--check-every", "25",
    "--layer-lr", "0.0002",
    "--teacher-alpha-start", "0.9",
    "--teacher-alpha-end", "0.0",
    "--accept-nmse", "0.2",
    "--accept-cosine", "0.9",
    "--accept-incremental-delta-nll", "0.015",
    "--accept-cumulative-delta-nll", "0.05",
    "--max-runtime-minutes", "240",
    "--resume", "--strict-acceptance",
]
run_env = dict(os.environ)
run_env["SEQUENTIAL_MAX_ROUNDS_PER_RUN"] = str(MAX_ROUNDS_PER_RUN)
run_env["PYTHONPATH"] = os.pathsep.join([str(REPO), str(REPO / "src")])
shell = "set -o pipefail; " + shlex.join(cmd) + " 2>&1 | tee -a " + shlex.quote(str(LOG))
print(shell, flush=True)
result = subprocess.run(["bash", "-lc", shell], cwd=REPO, env=run_env)
if result.returncode: raise RuntimeError(f"trainer failed with exit code {result.returncode}; see {LOG}")
print("✅ trainer returned normally (a scientific 'needs_more_training' status is not a crash)")


## Current status


In [ ]:
def show_json(name):
    p = OUT / name
    if p.exists():
        print(f"\n### {name}")
        print(p.read_text())
for name in ["sequential_run_status.json", "sequential_progress.json", "sequential_in_progress.json", "sequential_training_report.json"]:
    show_json(name)


## Original FunctionGemma vs accepted Memory Fusion snapshot

Only **accepted** replacements are loaded here. A currently-training unaccepted layer is deliberately excluded. The test uses deterministic greedy full-prefix decoding so the V1 Memory Fusion wrapper can run with `use_cache=False`.


In [ ]:
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.gemma3_memory_fusion import Gemma3MemoryFusionConfig, replace_attention_layers, structural_summary

device = torch.device("cuda")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
baseline = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=dtype, attn_implementation="sdpa").to(device).eval()
student = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=dtype, attn_implementation="sdpa").to(device).eval()

progress_path = OUT / "sequential_progress.pt"
accepted = []
if progress_path.exists():
    payload = torch.load(progress_path, map_location="cpu", weights_only=False)
    accepted = [int(x) for x in payload.get("accepted_layers", [])]
    mf_cfg = Gemma3MemoryFusionConfig.from_dict(payload["config"])
    if accepted:
        replace_attention_layers(student, mf_cfg, accepted)
        student.load_state_dict(payload["attention_state"], strict=False)
print("Accepted Memory Fusion layers:", accepted)
print("Structure:", json.dumps(structural_summary(student), indent=2))

@torch.no_grad()
def greedy_full_prefix(model, input_ids, max_new_tokens=40):
    seq_ids = input_ids.clone()
    for _ in range(max_new_tokens):
        logits = model(input_ids=seq_ids, use_cache=False, return_dict=True).logits[:, -1]
        nxt = logits.argmax(-1, keepdim=True)
        seq_ids = torch.cat([seq_ids, nxt], dim=1)
        if tok.eos_token_id is not None and int(nxt.item()) == int(tok.eos_token_id): break
    return seq_ids[:, input_ids.shape[1]:]

tools = [
 {"type":"function","function":{"name":"get_current_temperature","description":"Get temperature for a city","parameters":{"type":"object","properties":{"location":{"type":"string"}},"required":["location"]}}},
 {"type":"function","function":{"name":"calculate","description":"Calculate an expression","parameters":{"type":"object","properties":{"expression":{"type":"string"}},"required":["expression"]}}}
]
prompts = ["What's the temperature in Vienna?", "Calculate 18 times 7.", "What is the capital of Austria?"]
for prompt in prompts:
    messages = [{"role":"developer","content":"You are a model that can do function calling with the following functions"},{"role":"user","content":prompt}]
    enc = tok.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    ids = enc["input_ids"].to(device)
    a = greedy_full_prefix(baseline, ids)
    b = greedy_full_prefix(student, ids)
    A = tok.decode(a[0], skip_special_tokens=False)
    B = tok.decode(b[0], skip_special_tokens=False)
    print("\n" + "="*100)
    print("PROMPT:", prompt)
    print("\nORIGINAL FUNCTIONGEMMA:\n", A)
    print("\nMEMORY FUSION (accepted only):\n", B)
print("\n✅ tool/prompt smoke test complete")


## Archive this run


In [ ]:
archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Saved archive:", archive)
print("Persistent Drive folder:", OUT)
